# Single Source Shortest Path Problem 

In [ ]:
#Required Libraries

#run the following command in terminal to install the required libraries
#pip install networkx matplotlib imageio imageio-ffmpeg pandas

import networkx as nx
import matplotlib.pyplot as plt
import imageio
import os
import time
import heapq
from collections import deque
import pandas as pd
import random

import warnings
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
import random
import networkx as nx

# Global Configuration
NUM_NODES = 25
NUM_EDGES = 100  # Maintain sparse nature for better decluttering
source_node = 0   # Fixed source node for all algorithms

def generate_and_load_graph(num_nodes, num_edges):
    # 1. Generate a sparse random Directed Graph
    temp_g = nx.gnm_random_graph(num_nodes, num_edges, directed=True)
    
    # 2. Add random weights to edges
    edge_list = []
    for u, v in temp_g.edges():
        weight = random.randint(1, 20)
        edge_list.append({"Source": u, "Target": v, "weight": weight})
    
    # 3. Create a Spreadsheet (CSV) for Data Layer reference
    df = pd.DataFrame(edge_list)
    df.to_csv("graph_edges.csv", index=False)
    print(f"Success: 'graph_edges.csv' created with {num_nodes} nodes and {len(df)} edges.")
    
    # 4. Load with specific edge_attr mapping for existing algorithm code
    G = nx.from_pandas_edgelist(df, source="Source", target="Target", 
                                edge_attr="weight", 
                                create_using=nx.DiGraph())
    
    # Ensure all nodes exist in G (handling isolates)
    G.add_nodes_from(range(num_nodes))
    
    return G

# Initialize the optimized decluttered graph
G = generate_and_load_graph(NUM_NODES, NUM_EDGES)
metrics = {} # Reset metrics dictionary

In [ ]:
# Helper function to draw the graph state and save it as an image
def save_frame(graph, current_node, visited_edges, frame_idx, algo_name, final_path_edges=None):
    plt.figure(figsize=(14, 12)) # Larger figure size for better spacing
    
    # Spacing Control: Use high k (optimal distance between nodes) and fixed seed
    pos = nx.spring_layout(graph, seed=42, k=0.35, iterations=100) 
    
    # 1. Base Graph (Decluttered)
    nx.draw_networkx_nodes(graph, pos, node_size=30, node_color='black', alpha=0.8) # Small, faint nodes
    nx.draw_networkx_edges(graph, pos, alpha=0.3, edge_color='#dddddd') # Faint base edges
    
    # 2. Path Highlights: If final path is provided, ignore the blue traversal lines
    if final_path_edges:
        # Green = Optimized Shortest Path Tree Result
        nx.draw_networkx_edges(graph, pos, edgelist=final_path_edges, 
                                edge_color='green', width=2.5, alpha=1.0)
        plt.title(f"{algo_name} - FINAL SHORTEST PATH TREE (Frozen)")
    else:
        # Blue = Active Traversal Search Path
        nx.draw_networkx_edges(graph, pos, edgelist=visited_edges, 
                                edge_color='blue', width=1.5, alpha=0.7)
        plt.title(f"{algo_name} - Traversal Step {frame_idx}")
    
    # 3. Highlight current node being processed in red
    if current_node is not None:
        nx.draw_networkx_nodes(graph, pos, nodelist=[current_node], 
                                node_color='red', node_size=80)
    
    plt.axis('off') # Remove axis clutter
    filename = f"temp_{algo_name}_{frame_idx}.png"
    plt.savefig(filename, bbox_inches='tight')
    plt.close()
    return filename

# Helper function to compile video and delete frames (no changes required here)
def create_video_and_cleanup(frames, video_name, fps=3):
    with imageio.get_writer(video_name, fps=fps) as writer:
        for frame in frames:
            if os.path.exists(frame): # Safety check
                image = imageio.imread(frame)
                writer.append_data(image)
            
    # Cleanup temporary images
    for frame in frames:
        if os.path.exists(frame):
            os.remove(frame)
    print(f"Video saved as {video_name} and temporary images deleted.")


### Depth First Search (Brute Force)

In [ ]:
def dfs_sssp(graph, source):
    print("Running Exhaustive DFS (Brute Force SSSP)...")
    start_time = time.perf_counter()
    
    frames = []
    # Using a list for frame_idx so it can be mutated inside the nested recursive function
    frame_idx = [0] 
    
    distances = {node: float('inf') for node in graph.nodes}
    distances[source] = 0
    visited_edges = []
    predecessors = {node: None for node in graph.nodes}
    
    # Initial frame
    frames.append(save_frame(graph, source, visited_edges, frame_idx[0], "DFS_SSSP"))
    frame_idx[0] += 1
    
    def dfs(u, current_path_nodes, depth=0):
        if depth > 10: # Limits the search to 10 nodes deep
         return
    
        for v in graph.neighbors(u):
            if v not in current_path_nodes:  # Prevent infinite loops in cyclic graphs
                weight = graph[u][v]['weight']
                
                # Update distance if this specific path is shorter than previously found paths
                if distances[u] + weight < distances[v]:
                    distances[v] = distances[u] + weight
                    predecessors[v] = u
                
                # Visually traverse forward
                visited_edges.append((u, v))
                if frame_idx[0] % 30 == 0:  # Save a frame every 30 steps to reduce the number of frames
                 frames.append(save_frame(graph, v, visited_edges, frame_idx[0], "DFS_SSSP"))
                frame_idx[0] += 1
                
                # Recurse deeper
                current_path_nodes.add(v)
                dfs(v, current_path_nodes, depth + 1)
                
                # Backtrack: Remove node from path and edge from visualizer
                current_path_nodes.remove(v)
                visited_edges.pop() 
                
                # Frame to show the algorithm retreating (backtracking)
                frames.append(save_frame(graph, u, visited_edges, frame_idx[0], "DFS_SSSP"))
                frame_idx[0] += 1

    # Start the recursive exhaustive search
    dfs(source, {source})
    
    end_time = time.perf_counter()
    exec_time = end_time - start_time
    
    final_edges = [(predecessors[n], n) for n in graph.nodes if predecessors.get(n) is not None]
    
    # Increase range to 15, and pass an empty list [] instead of visited_edges
    for _ in range(8):
        # Notice the [0] added to frame_idx below!
        frames.append(save_frame(graph, None, [], frame_idx[0], "DFS", final_path_edges=final_edges))
        frame_idx[0] += 1  # Notice the [0] added here too!

        
    # Compile the video
    create_video_and_cleanup(frames, "DFS_Traversal.mp4")
    
    # Record metrics
    metrics['DFS (Brute Force)'] = {
        'Time Complexity': 'O(V!)', 
        'Space Complexity': 'O(V)', 
        'Exec Time (s)': exec_time
    }
    print(f"Exhaustive DFS finished in {exec_time:.6f} seconds.\n")

dfs_sssp(G, source_node)

### Breadth First Search (Brute Force)

In [ ]:
def bfs_sssp(graph, source):
    print("Running BFS SSSP...")
    start_time = time.perf_counter()
    
    frames = []
    frame_idx = 0
    visited = {source}
    queue = [source]
    visited_edges = []
    predecessors = {node: None for node in graph.nodes}
    
    
    frames.append(save_frame(graph, source, visited_edges, frame_idx, "BFS"))
    frame_idx += 1
    
    while queue:
        u = queue.pop(0)
        for v in graph.neighbors(u):
            if v not in visited:
                visited.add(v)
                visited_edges.append((u, v))
                predecessors[v] = u
                queue.append(v)
                
                
                frames.append(save_frame(graph, v, visited_edges, frame_idx, "BFS"))
                frame_idx += 1
                
    end_time = time.perf_counter()
    exec_time = end_time - start_time
    
    final_edges = [(predecessors[n], n) for n in graph.nodes if predecessors.get(n) is not None]
    
    # Increase range to 15, and pass an empty list [] instead of visited_edges
    for _ in range(8):
        frames.append(save_frame(graph, None, [], frame_idx, "BFS", final_path_edges=final_edges))
        frame_idx += 1
    
    create_video_and_cleanup(frames, "BFS_Traversal.mp4")
    
    metrics['BFS'] = {'Time Complexity': 'O(V + E)', 'Space Complexity': 'O(V)', 'Exec Time (s)': exec_time}
    print(f"BFS finished in {exec_time:.6f} seconds.\n")

bfs_sssp(G, source_node)

### Djikistra's Algorithm (Greedy Algorithm)

In [ ]:
def dijkstra_sssp(graph, source):
    print("Running Dijkstra's Algorithm...")
    start_time = time.perf_counter()
    
    frames = []
    frame_idx = 0
    
    distances = {node: float('inf') for node in graph.nodes}
    distances[source] = 0
    pq = [(0, source)]
    visited_edges = []
    predecessors = {node: None for node in graph.nodes}
    
    if frame_idx % 20 == 0:
     frames.append(save_frame(graph, source, visited_edges, frame_idx, "Dijkstra"))
    frame_idx += 1
    
    while pq:
        current_dist, u = heapq.heappop(pq)
        
        if current_dist > distances[u]:
            continue
            
        for v in graph.neighbors(u):
            weight = graph[u][v]['weight']
            distance = current_dist + weight
            
            if distance < distances[v]:
                distances[v] = distance
                predecessors[v] = u
                heapq.heappush(pq, (distance, v))
                visited_edges.append((u, v))
                
                frames.append(save_frame(graph, v, visited_edges, frame_idx, "Dijkstra"))
                frame_idx += 1
                
    end_time = time.perf_counter()
    exec_time = end_time - start_time
    
    final_edges = [(predecessors[n], n) for n in graph.nodes if predecessors.get(n) is not None]
    
    # Increase range to 15, and pass an empty list [] instead of visited_edges
    for _ in range(8):
        frames.append(save_frame(graph, None, [], frame_idx, "Dijkstra", final_path_edges=final_edges))
        frame_idx += 1

    
    
    create_video_and_cleanup(frames, "Dijkstra_Traversal.mp4")
    
    metrics['Dijkstra'] = {'Time Complexity': 'O(E + V log V)', 'Space Complexity': 'O(V)', 'Exec Time (s)': exec_time}
    print(f"Dijkstra finished in {exec_time:.6f} seconds.\n")

dijkstra_sssp(G, source_node)

### Bellman-Ford Algorithm (Dynamic Programming Approach)

In [ ]:
# def bellman_ford_sssp(graph, source):
#     print("Running Bellman-Ford Algorithm...")
#     start_time = time.perf_counter()
    
#     frames = []
#     frame_idx = 0
    
#     distances = {node: float('inf') for node in graph.nodes}
#     distances[source] = 0
#     visited_edges = []
#     predecessors = {node: None for node in graph.nodes}
    
#     if frame_idx % 20 == 0:
#      frames.append(save_frame(graph, source, visited_edges, frame_idx, "Bellman-Ford"))
#     frame_idx += 1
    
#     for _ in range(len(graph.nodes) - 1):
#         for u, v, data in graph.edges(data=True):
#             weight = data['weight']
#             if distances[u] + weight < distances[v]:
#                 distances[v] = distances[u] + weight
#                 predecessors[v] = u
#                 visited_edges.append((u, v))
                
#                 frames.append(save_frame(graph, v, visited_edges, frame_idx, "Bellman-Ford"))
#                 frame_idx += 1
                
#     end_time = time.perf_counter()
#     exec_time = end_time - start_time
#     final_edges = [(predecessors[n], n) for n in graph.nodes if predecessors.get(n) is not None]
    
#     # Increase range to 15, and pass an empty list [] instead of visited_edges
#     for _ in range(8):
#         frames.append(save_frame(graph, None, [], frame_idx, "Bellman-Ford", final_path_edges=final_edges))
#         frame_idx += 1
    
#     create_video_and_cleanup(frames, "Bellman_Ford_Traversal.mp4")
    
#     metrics['Bellman-Ford'] = {'Time Complexity': 'O(V * E)', 'Space Complexity': 'O(V)', 'Exec Time (s)': exec_time}
#     print(f"Bellman-Ford finished in {exec_time:.6f} seconds.\n")

# bellman_ford_sssp(G, source_node)

### Directed Acyclic Graph Shortest Path

In [ ]:
def dag_sssp(graph, source):
    print("Running DAG Shortest Path...")
    start_time = time.perf_counter()
    
    frames = []
    frame_idx = 0
    visited_edges = []
    
    # Topological sort
    topo_order = list(nx.topological_sort(graph))
    distances = {node: float('inf') for node in graph.nodes}
    distances[source] = 0
    predecessors = {node: None for node in graph.nodes}
    
    frames.append(save_frame(graph, source, visited_edges, frame_idx, "DAG_SSSP"))
    frame_idx += 1
    
    for u in topo_order:
        if distances[u] != float('inf'):
            for v in graph.neighbors(u):
                weight = graph[u][v]['weight']
                if distances[u] + weight < distances[v]:
                    distances[v] = distances[u] + weight
                    predecessors[v] = u
                    visited_edges.append((u, v))
                    
                    frames.append(save_frame(graph, v, visited_edges, frame_idx, "DAG_SSSP"))
                    frame_idx += 1

    end_time = time.perf_counter()
    exec_time = end_time - start_time
    final_edges = [(predecessors[n], n) for n in graph.nodes if predecessors.get(n) is not None]
    
    # Increase range to 15, and pass an empty list [] instead of visited_edges
    for _ in range(8):
        frames.append(save_frame(graph, None, [], frame_idx, "DAG", final_path_edges=final_edges))
        frame_idx += 1
    
    create_video_and_cleanup(frames, "DAG_SSSP_Traversal.mp4")
    
    metrics['DAG SSSP'] = {'Time Complexity': 'O(V + E)', 'Space Complexity': 'O(V)', 'Exec Time (s)': exec_time}
    print(f"DAG SSSP finished in {exec_time:.6f} seconds.\n")

# 1. Create a DAG version of your graph by ensuring edges only point "forward"
G_dag = G.copy()
# Remove any edge where the source index is greater than or equal to the target
# This is a mathematical trick to guarantee no cycles exist!
invalid_edges = [(u, v) for u, v in G_dag.edges() if u >= v]
G_dag.remove_edges_from(invalid_edges)

# 2. Check if the graph is empty (just in case)
if G_dag.number_of_edges() == 0:
    print("Warning: The DAG is empty! Try a different random seed or more edges.")
else:
    # 3. Run the algorithm on the new G_dag
    dag_sssp(G_dag, source_node)

### Shortest Path Faster Algorithm

In [ ]:
from collections import deque

def spfa_sssp(graph, source):
    print("Running SPFA...")
    start_time = time.perf_counter()
    
    frames = []
    frame_idx = 0
    visited_edges = []
    
    distances = {node: float('inf') for node in graph.nodes}
    distances[source] = 0
    queue = deque([source])
    in_queue = {node: False for node in graph.nodes}
    in_queue[source] = True
    predecessors = {node: None for node in graph.nodes}
    
    
    frames.append(save_frame(graph, source, visited_edges, frame_idx, "SPFA"))
    frame_idx += 1
    
    while queue:
        u = queue.popleft()
        in_queue[u] = False
        
        for v in graph.neighbors(u):
            weight = graph[u][v]['weight']
            if distances[u] + weight < distances[v]:
                distances[v] = distances[u] + weight
                predecessors[v] = u
                visited_edges.append((u, v))
                
                
                frames.append(save_frame(graph, v, visited_edges, frame_idx, "SPFA"))
                frame_idx += 1
                
                if not in_queue[v]:
                    queue.append(v)
                    in_queue[v] = True

    end_time = time.perf_counter()
    exec_time = end_time - start_time
    final_edges = [(predecessors[n], n) for n in graph.nodes if predecessors.get(n) is not None]
    
    # Increase range to 15, and pass an empty list [] instead of visited_edges
    for _ in range(8):
        frames.append(save_frame(graph, None, [], frame_idx, "SPFA", final_path_edges=final_edges))
        frame_idx += 1
    
    
    create_video_and_cleanup(frames, "SPFA_Traversal.mp4")
    
    metrics['SPFA'] = {'Time Complexity': 'O(E) Avg, O(V*E) Worst', 'Space Complexity': 'O(V)', 'Exec Time (s)': exec_time}
    print(f"SPFA finished in {exec_time:.6f} seconds.\n")

spfa_sssp(G, source_node)

## Comparison

In [ ]:
import pandas as pd

def print_comparison():
    print("\n--- Final Metrics Comparison ---")
    df = pd.DataFrame(metrics).T
    # Reorder columns for readability
    df = df[['Time Complexity', 'Space Complexity', 'Exec Time (s)']]
    display(df)

print_comparison()